# 🤖 05. Model Training & Experiments
### Amazon ML Challenge 2026: Multi-Source Business Entity Resolution

This notebook demonstrates the Gradient Boosting entity resolution model:
1. **Entity-Aware Split (`GroupShuffleSplit`):** All candidate pairs for any given Source 1 entity are assigned exclusively to either Train or Validation, strictly preventing cross-entity data leakage.
2. **LightGBM Classifier:** Configured for high-speed tabular ranking and probability estimation.
3. **Feature Importance:** Gain and split feature rankings.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "amazon" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json

import pandas as pd

from src.models.predict import load_matching_model
from src.utils.io import load_config

cfg = load_config(PROJECT_ROOT / "config.yaml")
models_dir = PROJECT_ROOT / cfg["paths"]["models_dir"]
model_path = models_dir / "matching_model.joblib"

if model_path.exists():
    model = load_matching_model(model_path)
    print(f"Loaded trained model: {type(model).__name__}")

    feat_path = models_dir / "feature_names.json"
    with open(feat_path, encoding="utf-8") as f:
        feats = json.load(f)
    print(f"Model uses {len(feats)} input features.")

## 1. Feature Importance Ranking
Identify the strongest predictive signals for resolving matching entities.

In [ ]:
feat_imp_path = PROJECT_ROOT / "artifacts" / "reports" / "feature_importance.csv"
if feat_imp_path.exists():
    df_imp = pd.read_csv(feat_imp_path)
    display(df_imp)
elif model_path.exists():
    from src.evaluation.feature_importance import extract_feature_importance

    df_imp = extract_feature_importance(model, feats)
    display(df_imp)